### Laboratorio 1

**Grupo 16**

**Integrantes:**
Mateo Zambrano 202321531,
Nicolas Romero

### 1. Configuración y carga de datos

### 1.1 Importación de librerías

In [7]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

### 1.2 Constantes de reproducibilidad

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.25
OBJETIVO = "temp_max_manana"

### 1.3 Carga de datos

In [8]:
DATA_DIR = Path("data")
RUTA_TRAIN = DATA_DIR / "datos Lab 1.csv"
RUTA_SIN_ETIQUETA = DATA_DIR / "Datos Test Lab 1.csv"
RUTA_DICCIONARIO = DATA_DIR / "Diccionario de datos.xlsx"

datos = pd.read_csv(RUTA_TRAIN)
sin_etuiqueta = pd.read_csv(RUTA_SIN_ETIQUETA)
diccionario = pd.read_excel(RUTA_DICCIONARIO)

print("Entrenamiento:", datos.shape)
print("No etiquetados: ", sin_etuiqueta.shape)
print("Columnas que están en train y no en el archivo sin etiquetar:", set(datos.columns) - set(sin_etuiqueta.columns))

diccionario

Entrenamiento: (2576, 27)
No etiquetados:  (364, 26)
Columnas que están en train y no en el archivo sin etiquetar: {'temp_max_manana'}


,variable,tipo,descripcion
0,fecha,texto,"Fecha de la observación, en formato día.mes.año."
1,presion_media,numérico,Presión atmosférica media del día (mbar).
2,presion_min,numérico,Presión atmosférica mínima del día (mbar).
3,presion_max,numérico,Presión atmosférica máxima del día (mbar).
4,presion_desv,numérico,Desviación típica de la presión durante el día (mbar).
5,humedad_media,numérico,Humedad relativa media del día (%).
6,humedad_min,numérico,Humedad relativa mínima del día (%).
7,humedad_max,numérico,Humedad relativa máxima del día (%).
8,humedad_desv,numérico,Desviación típica de la humedad relativa (%).
9,viento_media,numérico,Velocidad media del viento durante el día (m/s).


### 2. Exploración de datos

### 2.1 Estructura general/exploración inicial

In [10]:
display(datos.head())
datos.info()
datos.shape

,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,viento_min,viento_max,viento_desv,rafaga_media,rafaga_min,rafaga_max,rafaga_desv,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,0.05,1.559641,0.5753,1.3783,0.25,2.255577,0.8488,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,0.22,3.870000,0.9168,2.2274,0.63,6.130000,1.2544,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,0.12,3.640000,0.7299,2.0651,0.38,4.880000,0.9330,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,0.54,2.454415,0.7023,3.5649,1.38,4.430375,1.1835,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,1.00,7.810000,1.9521,5.9400,2.13,10.880000,NaN,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2576 entries, 0 to 2575
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              2504 non-null   object 
 1   presion_media      2501 non-null   float64
 2   presion_min        2502 non-null   float64
 3   presion_max        2512 non-null   float64
 4   presion_desv       2496 non-null   float64
 5   humedad_media      2502 non-null   float64
 6   humedad_min        2496 non-null   float64
 7   humedad_max        2490 non-null   float64
 8   humedad_desv       2515 non-null   float64
 9   viento_media       2490 non-null   float64
 10  viento_min         2485 non-null   float64
 11  viento_max         2495 non-null   float64
 12  viento_desv        2504 non-null   float64
 13  rafaga_media       2498 non-null   float64
 14  rafaga_min         2504 non-null   float64
 15  rafaga_max         2503 non-null   float64
 16  rafaga_desv        2497 

(2576, 27)

### 2.2 Observaciones

**fecha:** Se puede observar redundancias con las columnas **anio**, **dia_del_anio** y **mes**. En el caso de no coincidir, puede significar que algunos datos están corruptos, pues tienen información contradictoria. También se puede justificar un cambio de tipo de dato a DateTime64 con el fin de facilitar operaciones entre fechas.

**registros_del_dia:** Como número que representa la cantidad de mediciones de 10 minutos por día, es más justificable verlo como un int64 más que un float64.

**anio:** Redundante por la columna **fecha**. Este dato podría convertirse a int64 sin inconvenientes.

**dia_del_anio:** Redundante por la columna **fecha**. Este dato podría convertirse a int64 sin inconvenientes.

**estacion_anio:** Presenta datos que no corresponden a ninguna estación, como por ejemplo la fila 4. Estos valores inválidos se podrían reemplazar por datos NaN.

**mes:** Redundante por la columna **fecha**. Esta columna presenta datos escritos completamente en mayúsculas y otros datos en minúsculas.

**presion_media, min, max:** Son columnas que están correlacionadas, lo que puede afectar la interpretación de coeficientes.

**humedad_media, min, max:** Son columnas que están correlacionadas, lo que puede afectar la interpretación de coeficientes.

**viento_norte, viento_este, direccion_viento:** Son columnas que están correlacionadas, debido a que cada columna representa el componente de un mismo dato.

**sector_viento:** Debería tener 8 categorías (N, NE, E, SE, S, SO, O, NO) pero aparecen muchas más debido a variantes en mayúsculas, algunas en español o inglés en vez de la abreviatura, lo cual se podría considerar una inconsistencia.

**rafaga_min:** Presenta al menos un valor "centinela" (-9999), que representa un dato faltante codificado como número, por lo que debería convertirse a NaN antes alguna operación debido a que podría modificar la mediana o perjudicar la regresión lineal.

**humedad_media, humedad_min:** Se agrega esta observación aparte de **humedad_media, min, max** por su importancia: En estas columnas se observan 2 escalas mezcladas dentro de la misma columna, (una expresada como proporción 0-1 y otra como porcentaje 0%-100%). Esto es un problema de validez porque cada valor es válido, pero de forma separada. Se debería unificar a una sola escala.

### 2.3 Justificaciones

En esta sección se presentará la evidencia que sustenta las observaciones de la sección anterior, ordenadas siguiendo las cuatro dimensiones de calidad de datos (completitud, unicidad, consistencia, validez).

#### 2.3.1 Completitud

Mide si el dataset tiene toda la información esperada, o en otras palabras, que no falten datos.

In [14]:
(datos.isnull().sum()/datos.shape[0]).sort_values(ascending=False)

temp_max_manana      0.036879
viento_min           0.035326
mes                  0.034161
estacion_anio        0.034161
anio                 0.033385
humedad_max          0.033385
viento_media         0.033385
viento_max           0.031444
presion_desv         0.031056
humedad_min          0.031056
viento_norte         0.031056
rafaga_desv          0.030668
direccion_viento     0.030668
rafaga_media         0.030280
registros_del_dia    0.029891
presion_media        0.029115
humedad_media        0.028727
presion_min          0.028727
rafaga_max           0.028339
rafaga_min           0.027950
viento_desv          0.027950
fecha                0.027950
sector_viento        0.027562
dia_del_anio         0.026398
viento_este          0.024845
presion_max          0.024845
humedad_desv         0.023680
dtype: float64

In [ ]:
print("faltantes por año")
for anio in datos["anio"].dropna().unique():
   datos_anio = datos[datos["anio"] == anio]
   n_faltantes = datos_anio.isnull().sum().sum()
   print(f"{int(anio)}: {n_faltantes}")

faltantes por año
2009: 257
2010: 273
2011: 300
2012: 265
2013: 275
2014: 260
2015: 306


**Valores faltantes:** Todas las columnas presentan un porcentaje de datos faltantes de entre 2%-4%, lo que sugiere que no hay un patrón de ausencia. Como es un porcentaje bajo y disperso, se imputarán los valores con la mediana en lugar de eliminar las filas para poder minimizar la pérdida de datos para el modelo.

#### 2.3.2 Unicidad

Verifica que cada registro aparece exactamente 1 vez, por lo que se buscarán detectar filas idénticas y duplicados lógicos (misma llave de negocio con datos diferentes). En este caso, la llave de negocio es la fecha, por lo que cada día deberá aparecer una sola vez. 

In [ ]:
#Filas identicas
print("Duplicados exactos:", datos.duplicated(keep=False).sum())

#Duplicados lógicos
fechas_repetidas = datos["fecha"].value_counts().loc[lambda x: x > 1]
print("Fechas que se repiten:", len(fechas_repetidas))